# Module 03 — Model-Free RL: Monte Carlo & Temporal-Difference

**Reinforcement Learning · Dakar Institute of Technology (DIT)**

In Module 02 we *knew* the model. Usually we don't. **Model-free** methods learn
**directly from experience** — sampled episodes of $(s, a, r, s')$ — without ever
knowing $P$ or $R$.

Two big families:
- **Monte Carlo (MC):** learn from **complete episodes**; update toward the actual
  return $G_t$.
- **Temporal-Difference (TD):** learn from **single steps** by **bootstrapping** —
  update toward $r + \gamma V(s')$ (an estimate), no need to wait for the episode
  to end.

We'll cover: **MC prediction & control**, **TD(0) prediction**, then the two
classic TD **control** algorithms — **SARSA** (on-policy) and **Q-learning**
(off-policy) — and the famous CliffWalking comparison.

### Learning objectives
1. Estimate $V^\pi$ with **first-visit Monte Carlo**.
2. Find $\pi^*$ with **on-policy MC control** ($\varepsilon$-greedy).
3. Estimate $V^\pi$ with **TD(0)** and contrast MC vs TD.
4. Implement **SARSA** and **Q-learning**.
5. Understand *why* SARSA and Q-learning behave differently near a cliff.

In [ ]:
import sys, pathlib
ROOT = pathlib.Path.cwd()
while not (ROOT / "rl_helpers").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict
from rl_helpers import (make_default_gridworld, set_seed, epsilon_greedy,
                        greedy_policy_from_Q, plot_state_values, plot_policy,
                        plot_learning_curve, animate_policy)

set_seed(0)
env = make_default_gridworld(slip=0.1, gamma=0.99)
rng = np.random.default_rng(0)
print(env.render_ascii())

## 1. Monte Carlo prediction (first-visit)

Estimate $V^\pi(s)$ = average return following the first visit to $s$ across many
episodes. We evaluate the **uniform-random policy** so we can compare against the
exact DP value from Module 02.

$$V(s) \leftarrow \text{average of } G_t \text{ over first visits to } s$$

In [ ]:
def generate_episode(env, policy_fn, max_steps=200):
    s = env.reset()
    episode = []
    for _ in range(max_steps):
        a = policy_fn(s)
        ns, r, done, _ = env.step(a)
        episode.append((s, a, r))
        s = ns
        if done:
            break
    return episode

def mc_prediction(env, policy_fn, n_episodes=20_000, gamma=None):
    gamma = env.gamma if gamma is None else gamma
    returns_sum = defaultdict(float)
    returns_cnt = defaultdict(int)
    V = np.zeros(env.n_states)
    for _ in range(n_episodes):
        episode = generate_episode(env, policy_fn)
        G = 0.0
        visited = set()
        for t in reversed(range(len(episode))):
            s, a, r = episode[t]
            # TODO: update the return G = r + gamma * G
            # TODO: if this is the FIRST visit to s in the episode, update the
            # running average V[s].
            raise NotImplementedError("Implement first-visit MC prediction")
    return V

random_pol = lambda s: int(rng.integers(env.n_actions))
V_mc = mc_prediction(env, random_pol, n_episodes=20_000)
plot_state_values(env, V_mc, "V of random policy (Monte Carlo estimate)")
plt.show()

## 2. Monte Carlo control ($\varepsilon$-greedy)

To *improve* the policy we need action values $Q(s,a)$. On-policy MC control:
generate episodes with an $\varepsilon$-greedy policy over the current $Q$, then
update $Q$ toward the observed returns. Over time we **decay $\varepsilon$** so the
policy becomes (nearly) greedy.

In [ ]:
def mc_control(env, n_episodes=50_000, gamma=None, eps_start=1.0, eps_end=0.05):
    gamma = env.gamma if gamma is None else gamma
    Q = np.zeros((env.n_states, env.n_actions))
    counts = np.zeros((env.n_states, env.n_actions))
    ep_returns = []
    for ep in range(n_episodes):
        eps = max(eps_end, eps_start - (eps_start - eps_end) * ep / (0.8 * n_episodes))
        s = env.reset()
        episode, G = [], 0.0
        for _ in range(200):
            # TODO: pick an eps-greedy action (use epsilon_greedy(Q[s], eps, ...))
            # TODO: step the env, append (s, a, r), advance s, break if done
            raise NotImplementedError("Generate the episode")
        # TODO: first-visit MC update of Q using an incremental average
        # Q[s,a] += (G - Q[s,a]) / counts[s,a]
        ep_returns.append(sum(r for _, _, r in episode))
    return Q, ep_returns

Q_mc, mc_returns = mc_control(env, n_episodes=50_000)
policy_mc = greedy_policy_from_Q(Q_mc)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
plot_policy(env, policy_mc, "MC-control greedy policy", ax=axes[0])
plot_learning_curve(mc_returns, window=500, title="MC control learning curve", ax=axes[1])
plt.tight_layout(); plt.show()

## 3. TD(0) prediction — bootstrapping

MC waits for the whole episode. **TD(0)** updates after *every step* using the
**TD target** $r + \gamma V(s')$:
$$V(s) \leftarrow V(s) + \alpha\,\big[\underbrace{r + \gamma V(s')}_{\text{TD target}} - V(s)\big]$$
The bracket is the **TD error** $\delta$. TD can learn online, from incomplete
episodes, and usually with lower variance than MC.

In [ ]:
def td0_prediction(env, policy_fn, n_episodes=20_000, alpha=0.1, gamma=None):
    gamma = env.gamma if gamma is None else gamma
    V = np.zeros(env.n_states)
    for _ in range(n_episodes):
        s = env.reset()
        for _ in range(200):
            a = policy_fn(s)
            ns, r, done, _ = env.step(a)
            # TODO: compute the TD target r + gamma * V[ns] (0 future value if done)
            # TODO: update V[s] toward the TD target with step size alpha
            raise NotImplementedError("Implement the TD(0) update")
            s = ns
            if done:
                break
    return V

V_td = td0_prediction(env, random_pol, n_episodes=20_000, alpha=0.05)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
plot_state_values(env, V_mc, "Monte Carlo V", ax=axes[0])
plot_state_values(env, V_td, "TD(0) V", ax=axes[1])
plt.tight_layout(); plt.show()

## 4. SARSA — on-policy TD control

SARSA updates $Q$ using the action **actually taken** next (the name comes from the
tuple $S, A, R, S', A'$):
$$Q(s,a) \leftarrow Q(s,a) + \alpha\big[r + \gamma\,Q(s',a') - Q(s,a)\big]$$
It is **on-policy**: it evaluates the same $\varepsilon$-greedy policy it follows.

In [ ]:
def sarsa(env, n_episodes=10_000, alpha=0.1, gamma=None, eps=0.1):
    gamma = env.gamma if gamma is None else gamma
    Q = np.zeros((env.n_states, env.n_actions))
    ep_returns = []
    for _ in range(n_episodes):
        s = env.reset()
        a = epsilon_greedy(Q[s], eps, env.n_actions, rng)
        G = 0.0
        for _ in range(200):
            ns, r, done, _ = env.step(a)
            # TODO: choose next action na eps-greedily from Q[ns]
            # TODO: SARSA target uses Q[ns, na]; update Q[s, a]
            # TODO: advance (s, a) <- (ns, na)
            raise NotImplementedError("Implement the SARSA update")
            if done:
                break
        ep_returns.append(G)
    return Q, ep_returns

Q_sarsa, sarsa_returns = sarsa(env, n_episodes=10_000)
plot_policy(env, greedy_policy_from_Q(Q_sarsa), "SARSA greedy policy")
plt.show()

## 5. Q-learning — off-policy TD control

Q-learning uses the **greedy** action in the target (a `max`), regardless of what
it actually did next — so it learns the **optimal** policy while *behaving*
$\varepsilon$-greedily:
$$Q(s,a) \leftarrow Q(s,a) + \alpha\big[r + \gamma\,\max_{a'} Q(s',a') - Q(s,a)\big]$$

In [ ]:
def q_learning(env, n_episodes=10_000, alpha=0.1, gamma=None, eps=0.1):
    gamma = env.gamma if gamma is None else gamma
    Q = np.zeros((env.n_states, env.n_actions))
    ep_returns = []
    for _ in range(n_episodes):
        s = env.reset()
        G = 0.0
        for _ in range(200):
            a = epsilon_greedy(Q[s], eps, env.n_actions, rng)
            ns, r, done, _ = env.step(a)
            # TODO: Q-learning target uses max_a' Q[ns, a'] (0 if done). Update Q[s,a].
            raise NotImplementedError("Implement the Q-learning update")
            s = ns
            G += r
            if done:
                break
        ep_returns.append(G)
    return Q, ep_returns

Q_ql, ql_returns = q_learning(env, n_episodes=10_000)
plot_policy(env, greedy_policy_from_Q(Q_ql), "Q-learning greedy policy")
plt.show()
animate_policy(env, lambda s: int(np.argmax(Q_ql[s])), max_steps=40, fps=3,
               path="qlearning_optimal.gif")

## 6. SARSA vs Q-learning: the Cliff

The classic experiment (Sutton & Barto, Example 6.6). In **CliffWalking**, the
agent walks along the edge of a cliff. Falling off costs **−100**.

- **Q-learning** learns the *optimal* path — right along the cliff edge — but
  because it still *explores* $\varepsilon$-greedily, it occasionally falls in, so
  its **online** returns are worse.
- **SARSA** accounts for exploration in its target, so it learns a *safer* path one
  row away from the cliff, earning **higher returns during training**.

We use Gymnasium's `CliffWalking-v1` (a tabular env with 48 states, 4 actions).

In [ ]:
import gymnasium as gym

def train_tabular(algo, env_id="CliffWalking-v1", n_episodes=500, alpha=0.5,
                  gamma=1.0, eps=0.1):
    e = gym.make(env_id)
    nS, nA = e.observation_space.n, e.action_space.n
    Q = np.zeros((nS, nA))
    ep_returns = []
    for _ in range(n_episodes):
        s, _ = e.reset()
        a = epsilon_greedy(Q[s], eps, nA, rng)
        G, done = 0.0, False
        while not done:
            ns, r, term, trunc, _ = e.step(a)
            done = term or trunc
            na = epsilon_greedy(Q[ns], eps, nA, rng)
            if algo == "sarsa":
                target = r + gamma * Q[ns, na] * (not done)
            else: # q-learning
                target = r + gamma * np.max(Q[ns]) * (not done)
            Q[s, a] += alpha * (target - Q[s, a])
            s, a = ns, na
            G += r
        ep_returns.append(G)
    e.close()
    return Q, ep_returns

_, sarsa_cliff = train_tabular("sarsa")
_, qlearn_cliff = train_tabular("qlearning")

fig, ax = plt.subplots(figsize=(8, 4))
plot_learning_curve(sarsa_cliff, window=20, title="CliffWalking: SARSA vs Q-learning",
                    ax=ax, label="SARSA")
plot_learning_curve(qlearn_cliff, window=20, ax=ax, label="Q-learning")
ax.set_ylim(-200, 0)
plt.show()
print("Q-learning finds the OPTIMAL (risky) path; SARSA finds the SAFER path.")

## Recap
| Method | Learns from | Target | On/Off-policy |
|---|---|---|---|
| Monte Carlo | full episodes | actual return $G_t$ | on-policy |
| TD(0) | single steps | $r+\gamma V(s')$ | on-policy |
| SARSA | single steps | $r+\gamma Q(s',a')$ | **on**-policy |
| Q-learning | single steps | $r+\gamma \max_{a'}Q(s',a')$ | **off**-policy |

### Going further (optional, same environments)
- On the GridWorld, compare the MC-control and Q-learning greedy policies — do they
  agree? Try different `alpha` and `eps` and watch the learning curves change.
- On CliffWalking, vary `eps`: as exploration goes to zero, SARSA's safe path should
  converge toward Q-learning's optimal (risky) path.

**Next:** Module 04 — what if the state space is too big for a table? Function
approximation.